In [1]:
#Установка библиотек
!pip install -q transformers evaluate accelerate
!pip install -q --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.9 MB/s eta 0:00:00


In [2]:

import torch
import numpy as np
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
import evaluate
from google.colab import userdata
from huggingface_hub import login

# Загрузка датасета RUSSE из RussianSuperGLUE
login(token=userdata.get('HF_TOKEN'))
model_name = "DeepPavlov/rubert-base-cased"

# Загрузка датасета из локальных jsonl файлов
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl"
})

tokenizer = AutoTokenizer.from_pretrained(model_name)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [3]:
# Функция препроцессинга с явным выделением целевого слова
def preprocess_function(examples):
    first_sentences = [
        f"Слово: {w}. Контекст 1: {s1}"
        for w, s1 in zip(examples["word"], examples["sentence1"])
    ]
    second_sentences = [
        f"Контекст 2: {s2}"
        for s2 in examples["sentence2"]
    ]

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=128,
        padding=False
    )

    # Безопасное извлечение меток
    raw_labels = examples.get("label", examples.get("state", []))

    # конвертация любых форматов
    tokenized["label"] = [1 if str(l).lower() in ['true', '1'] else 0 for l in raw_labels]

    return tokenized

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset = tokenized_dataset.cast_column("label", Value("int64"))

# Фиксация формата колонок
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "token_type_ids", "attention_mask", "label"]
)


Map:   0%|          | 0/19845 [00:00<?, ? examples/s]

Map:   0%|          | 0/8505 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/19845 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8505 [00:00<?, ? examples/s]

In [4]:
# Метрика Accuracy
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [5]:
# Инициализация модели
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    problem_type="single_label_classification"  # Включение CrossEntropyLoss
)



pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  714MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

In [9]:
# Гиперпараметры обучения
training_args = TrainingArguments(
    output_dir="./rubert_russe_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none"
)



In [10]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Старт обучения
print("Начинаем обучение...")
trainer.train()


Начинаем обучение...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:

# Финальная оценка
eval_results = trainer.evaluate()
print("Validation Accuracy:", eval_results["eval_accuracy"])

Training Loss,Validation Loss,Epoch,Accuracy
0.043507,0.380199,4,0.936508


Validation Accuracy: 0.9365079365079365


In [ ]:
# Архивируем всю папку с результатами
!zip -r rubert_russe_results.zip /content/rubert_russe_results

# Запускаем скачивание архива
from google.colab import files
files.download('rubert_russe_results.zip')